# PDAC internal val — pretrained pan-cancer Hist2Pheno

CODEX PDAC has **56 fold-0** coverslip regions. This notebook only draws the
**top 5** by matched-cell **L2 macro AUROC** among those sections. Individual-dataset notebook:
`code/CODEX_pdac/Data_process_visual_codex_HEcelltype_pdac.ipynb`.

Checkpoint: `data/PanCancer/result_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42/cross_dataset_cv/A2_adjacent_spatial_k8mean/best_mlp_gpu.pt`  
This is the UNI2 A2 best model. That file is the fold-1 checkpoint, so these notebooks show **fold 0** instead of fold 1. Deployment skipped the 4×3 panel PNGs, so GT versus prediction is drawn from `matched_pred.h5ad`.

```bash
bash code/Internal_val/demo.sh status
ONLY=PDAC bash code/Internal_val/demo.sh predict
```


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
if not (REPO / "code").is_dir():
    REPO = Path("/nobackup2/users/lingyu/Python/Hist2Pheno")
sys.path.insert(0, str(REPO / "code" / "Internal_val"))

import internal_val as iv

DATASET = "PDAC"
SAVE_FIGURES = False
SAVE_PAPER = True
fold = iv.fold0_table(DATASET)
viz = iv.top_fold0_samples(DATASET, 5)
rank = iv.rank_fold0(DATASET)
cols = [
    "rank_ds", "n_ds", "rank_all", "n_all",
    "sample", "n_matched",
    "macro_auc_L1_cells", "macro_auc_L12_cells", "macro_auc_L2_cells",
]
rank_show = rank[cols].copy()
print(f"{DATASET} fold {iv.FOLD}: {len(fold)} sections; drawing top {len(viz)} by L2 AUROC")
print("top5", viz)
display(Markdown("### Top 5 PDAC fold-0 by L2 AUROC"))
display(rank_show.head(5))
display(Markdown("### All 56 fold-0 sections (rank table; notebook does not draw the rest)"))
display(rank_show)


In [ ]:
display(Markdown("## Top 5 — 4×3 panels (GT / Hist2Pheno / ROC / StarDist-all)"))
missing = []
for sample in viz:
    panel = iv.pred_paths(DATASET, sample)["panel"]
    display(Markdown(f"### {iv.rank_label(DATASET, sample)}"))
    if panel.is_file():
        display(Image(filename=str(panel)))
    else:
        missing.append(sample)
        print("missing panel", sample)
if missing:
    print("UNI2 A2 deployment did not save 4x3 panel PNGs. GT versus prediction is drawn from matched_pred.h5ad.")


In [ ]:
display(Markdown("## Top 5 — GT vs pretrained L2 (matched nuclei)"))
for sample in viz:
    pred = iv.pred_paths(DATASET, sample)["pred"]
    if not pred.is_file():
        print("skip", sample, "(no matched_pred.h5ad)")
        continue
    display(Markdown(f"### {iv.rank_label(DATASET, sample)}"))
    fig = iv.plot_spatial_gt_pred(DATASET, sample, "L2", save=SAVE_FIGURES)
    display(fig)
    plt.close(fig)
    fig = iv.plot_composition(DATASET, sample, "L2", save=SAVE_FIGURES)
    display(fig)
    plt.close(fig)

if SAVE_PAPER:
    out = iv.export_paper_sections(DATASET, viz)
    display(Markdown(f"Saved paper figures → `{out}`"))
